In [ ]:
"""
The next best augmentation to test is:
    ==> Amplitude Scaling

We will keep everything the same as in Phase 3 Gaussian noise, and
only change the augmentation type.

That means:

- same subject
- same folds
- same baseline EEGNet
- same training pipeline
- same evaluation method

Only the augmentation changes.

Amplitude scaling means: "multiply the EEG signal by a small random factor"

For example:
  -> one trial may be multiplied by 0.95
  -> another by 1.03
  -> another by 1.08
So the shape of the signal stays similar,
but the overall strength becomes slightly lower or higher.
It is like saying:
“What if the same brain pattern appears a little weaker or a little stronger?”
This is realistic because EEG amplitude can vary naturally.

This is useful because,
 model should ideally learn the class even if the signal amplitude changes a little.

So amplitude scaling helps the model become less sensitive to small signal-strength differences.

"""

'\nThe next best augmentation to test is:\n    ==> Amplitude Scaling\n\nWe will keep everything the same as in Phase 3 Gaussian noise, and \nonly change the augmentation type.\n\nThat means:\n\n- same subject\n- same folds\n- same baseline EEGNet\n- same training pipeline\n- same evaluation method\n\nOnly the augmentation changes.\n\nAmplitude scaling means: "multiply the EEG signal by a small random factor"\n\nFor example:\n  -> one trial may be multiplied by 0.95\n  -> another by 1.03\n  -> another by 1.08\nSo the shape of the signal stays similar, \nbut the overall strength becomes slightly lower or higher.\nIt is like saying:\n“What if the same brain pattern appears a little weaker or a little stronger?”\nThis is realistic because EEG amplitude can vary naturally.\n\nThis is useful because,\n model should ideally learn the class even if the signal amplitude changes a little.\n\nSo amplitude scaling helps the model become less sensitive to small signal-strength differences.\n\n'

In [ ]:
# ---------------------------------------
# step 1 : Mount Google Drive
# ---------------------------------------
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ---------------------------------------
# step 2: importing the required libraries
# ---------------------------------------

import os
import random
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, train_test_split

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, DepthwiseConv2D
from tensorflow.keras.layers import AveragePooling2D, SeparableConv2D
from tensorflow.keras.layers import BatchNormalization, Activation
from tensorflow.keras.layers import Dropout, Flatten, Dense
from tensorflow.keras.constraints import max_norm
from tensorflow.keras.utils import to_categorical

In [ ]:
# ---------------------------------------
# step 3 : Basic experiment settings
# ---------------------------------------


random_seed = 42
subject_file_name = "S14_EEG.mat"

dataset_folder = "/content/drive/MyDrive/Colab_Notebooks/My_Project/imagined_speech/Base de Datos Habla Imaginada/S14"
results_folder = "/content/drive/MyDrive/Colab_Notebooks/My_Project/Results_April2026"

number_of_signal_columns = 24576
number_of_channels = 6
number_of_samples_per_trial = 4096
number_of_folds = 5

batch_size = 16
number_of_epochs = 50
validation_size_inside_training = 0.2

In [ ]:
# ---------------------------------------
# step 4 : make results folder and fix randomness
# ---------------------------------------

# Creates the results folder if it doesn't exist
os.makedirs(results_folder, exist_ok=True)

np.random.seed(random_seed)
random.seed(random_seed)
tf.random.set_seed(random_seed)
print("==============================================")
print("Results folder is ready.")
print("Random seed is set.")
print("==============================================")

Results folder is ready.
Random seed is set.


In [ ]:
# ---------------------------------------
# step 5 : building the subject path and load the file
# ---------------------------------------

import scipy.io as sio

subject_file_name = "S14_EEG.mat"

subject_file_path = os.path.join(dataset_folder, subject_file_name)

print("==============================================")
print("Subject file path:", subject_file_path)
print("Does file exist?", os.path.exists(subject_file_path))
print("==============================================")
mat_data = sio.loadmat(subject_file_path)
print("==============================================")
print("\nThe .mat file loaded successfully.")
print("Available keys inside the file:", mat_data.keys())
print("==============================================")

Subject file path: /content/drive/MyDrive/Colab_Notebooks/My_Project/imagined_speech/Base de Datos Habla Imaginada/S14/S14_EEG.mat
Does file exist? True

The .mat file loaded successfully.
Available keys inside the file: dict_keys(['__header__', '__version__', '__globals__', 'EEG'])


In [ ]:
# ---------------------------------------
# step 6 : extract EEG matrix and printing shape
# ---------------------------------------

eeg_matrix = mat_data["EEG"]
print("==============================================")
print("EEG matrix extracted.")
print("number_of_trials, number_of_columns are : ")
print("==============================================")
print("EEG matrix shape:", eeg_matrix.shape)
print("==============================================")

EEG matrix extracted.
number_of_trials, number_of_columns are : 
EEG matrix shape: (639, 24579)


In [ ]:
# ---------------------------------------
# step 7 : separating the signal data and metadata columns
# ---------------------------------------

number_of_signal_columns = 24576

signal_data = eeg_matrix[:, :number_of_signal_columns]
modality_column = eeg_matrix[:, number_of_signal_columns]
stimulus_column = eeg_matrix[:, number_of_signal_columns + 1]
artifact_column = eeg_matrix[:, number_of_signal_columns + 2]

print("==============================================")
print("Signal data shape:", signal_data.shape)
print("Modality column shape:", modality_column.shape)
print("Stimulus column shape:", stimulus_column.shape)
print("Artifact column shape:", artifact_column.shape)
print("==============================================")

print("Unique modality values:", np.unique(modality_column))
print("Unique stimulus values:", np.unique(stimulus_column))
print("Unique artifact values:", np.unique(artifact_column))
print("==============================================")


Signal data shape: (639, 24576)
Modality column shape: (639,)
Stimulus column shape: (639,)
Artifact column shape: (639,)
Unique modality values: [1. 2.]
Unique stimulus values: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11.]
Unique artifact values: [1. 2.]


In [ ]:
# ---------------------------------------
# step 8: filtering only imagined speech and valid trials
# ---------------------------------------

valid_trial_mask = (modality_column == 1) & (artifact_column == 1)

filtered_signal_data = signal_data[valid_trial_mask]
filtered_labels = stimulus_column[valid_trial_mask]

print("==============================================")
print("Number of valid filtered trials:", len(filtered_labels))
print("Filtered signal shape:", filtered_signal_data.shape)
print("Filtered labels shape:", filtered_labels.shape)
print("==============================================")

print("Unique filtered labels:", np.unique(filtered_labels))
print("==============================================")

Number of valid filtered trials: 351
Filtered signal shape: (351, 24576)
Filtered labels shape: (351,)
Unique filtered labels: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11.]


In [ ]:
# ---------------------------------------
# step 9: reshape filtered signal into trial format
# ---------------------------------------


number_of_channels = 6
number_of_samples_per_trial = 4096

X = filtered_signal_data.reshape(-1, number_of_channels, number_of_samples_per_trial)
y = filtered_labels.astype(int)

print("==============================================")
print("X shape after reshape:", X.shape)
print("y shape:", y.shape)
print("==============================================")

X shape after reshape: (351, 6, 4096)
y shape: (351,)


In [ ]:
# ---------------------------------------
# step 10 : prepare labels for classification - shift labels to 0 ... 10
# ---------------------------------------

y = y - 1

number_of_classes = len(np.unique(y))
label_counts = pd.Series(y).value_counts().sort_index()

print("==============================================")
print("Unique labels after shifting to 0-based indexing:")
print(np.unique(y))
print("==============================================")

print("Number of classes:", number_of_classes)
print("==============================================")

print("Class counts:")
print(label_counts)
print("==============================================")

Unique labels after shifting to 0-based indexing:
[ 0  1  2  3  4  5  6  7  8  9 10]
Number of classes: 11
Class counts:
0     35
1     40
2     37
3     33
4     34
5     28
6     32
7     31
8     27
9     25
10    29
Name: count, dtype: int64


In [ ]:
# ---------------------------------------
# step 11: creating 5-fold stratified cross-validation splits
# ---------------------------------------

from sklearn.model_selection import StratifiedKFold

number_of_folds = 5
random_seed = 42

stratified_kfold = StratifiedKFold(
    n_splits=number_of_folds, shuffle=True, random_state=random_seed)

fold_splits = list(stratified_kfold.split(X, y))

print("==============================================")
print("Total number of folds created:", len(fold_splits))
print("==============================================")

for fold_number, (train_indices, test_indices) in enumerate(fold_splits, start=1):
    print("Fold", fold_number)
    print("Train size:", len(train_indices))
    print("Test size:", len(test_indices))
    print("----------------------------------------------")


Total number of folds created: 5
Fold 1
Train size: 280
Test size: 71
----------------------------------------------
Fold 2
Train size: 281
Test size: 70
----------------------------------------------
Fold 3
Train size: 281
Test size: 70
----------------------------------------------
Fold 4
Train size: 281
Test size: 70
----------------------------------------------
Fold 5
Train size: 281
Test size: 70
----------------------------------------------


In [ ]:
# ==============================================
# Baseline EEGNet model (needed for Phase 4)
# ==============================================

def build_baseline_eegnet(input_shape, number_of_classes):
    input_layer = Input(shape=input_shape)

    block_1 = Conv2D(
        filters=8,
        kernel_size=(1, 64),
        padding="same",
        use_bias=False
    )(input_layer)
    block_1 = BatchNormalization()(block_1)

    block_1 = DepthwiseConv2D(
        kernel_size=(input_shape[0], 1),
        depth_multiplier=2,
        use_bias=False,
        depthwise_constraint=max_norm(1.0)
    )(block_1)
    block_1 = BatchNormalization()(block_1)
    block_1 = Activation("elu")(block_1)
    block_1 = AveragePooling2D(pool_size=(1, 4))(block_1)
    block_1 = Dropout(0.5)(block_1)

    block_2 = SeparableConv2D(
        filters=16,
        kernel_size=(1, 16),
        padding="same",
        use_bias=False
    )(block_1)
    block_2 = BatchNormalization()(block_2)
    block_2 = Activation("elu")(block_2)
    block_2 = AveragePooling2D(pool_size=(1, 8))(block_2)
    block_2 = Dropout(0.5)(block_2)

    flatten_layer = Flatten()(block_2)

    output_layer = Dense(
        units=number_of_classes,
        activation="softmax",
        kernel_constraint=max_norm(0.25)
    )(flatten_layer)

    model = Model(inputs=input_layer, outputs=output_layer)

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In Phase 4, we will use: amplitude scaling

In [ ]:
# Define the amplitude scaling augmentation function


# ==============================================
# Phase 4 - Step 1: amplitude scaling augmentation
# ==============================================

def apply_amplitude_scaling(data_array, min_scale=0.9, max_scale=1.1):
    scaling_factors = np.random.uniform(
        low=min_scale,
        high=max_scale,
        size=(data_array.shape[0], 1, 1)
    )

    scaled_data = data_array * scaling_factors
    return scaled_data

In [ ]:
sample_data = X[:2]
sample_scaled_data = apply_amplitude_scaling(sample_data, min_scale=0.9, max_scale=1.1)

print("Original shape:", sample_data.shape)
print("Scaled shape:", sample_scaled_data.shape)

Original shape: (2, 6, 4096)
Scaled shape: (2, 6, 4096)


In [ ]:
"""
As per the above code,
  apply_amplitude_scaling is a function

def apply_amplitude_scaling(...)
This creates a function whose job is:
take EEG trials and slightly increase or decrease their amplitude

min_scale=0.9, max_scale=1.1
This means:
  -> some trials may be multiplied by a value slightly below 1
  -> some by a value slightly above 1

So the amplitude may become:
  -> a little smaller or a little larger

Examples:

0.92
1.03
1.08
These are small changes, which is what we want.

size=(data_array.shape[0], 1, 1)
This is very important.

It means:
generate one scaling factor per trial
apply that same factor to all channels and all time samples of that trial

So each trial gets its own overall amplitude adjustment.
That is a clean and simple form of amplitude scaling.

scaled_data = data_array * scaling_factors
This applies the scaling.
So if one trial has a scaling factor of 1.05, the whole trial becomes 5% stronger.
If one trial has 0.95, the whole trial becomes 5% weaker.
The class label stays the same.

this is useful because:
“Do not rely too heavily on exact signal strength.
Learn the pattern even if the amplitude changes a little.”
That can help generalization.

"""

'\nAs per the above code, \n  apply_amplitude_scaling is a function\n\ndef apply_amplitude_scaling(...)\nThis creates a function whose job is:\ntake EEG trials and slightly increase or decrease their amplitude\n\nmin_scale=0.9, max_scale=1.1\nThis means:\n  -> some trials may be multiplied by a value slightly below 1\n  -> some by a value slightly above 1\n\nSo the amplitude may become:\n  -> a little smaller or a little larger\n\nExamples:\n\n0.92\n1.03\n1.08\nThese are small changes, which is what we want.\n\nsize=(data_array.shape[0], 1, 1)\nThis is very important.\n\nIt means:\ngenerate one scaling factor per trial\napply that same factor to all channels and all time samples of that trial\n\nSo each trial gets its own overall amplitude adjustment.\nThat is a clean and simple form of amplitude scaling.\n\nscaled_data = data_array * scaling_factors\nThis applies the scaling.\nSo if one trial has a scaling factor of 1.05, the whole trial becomes 5% stronger.\nIf one trial has 0.95, th

In [ ]:
# Creating the fold training function with amplitude scaling



# ==============================================
# Phase 4 - Step 2: one-fold training function
# with amplitude scaling augmentation
# ==============================================

def run_one_fold_with_amplitude_scaling(X, y, train_indices, test_indices, fold_number, min_scale=0.9, max_scale=1.1):
    # ------------------------------------------
    # Step 2.1: create training and test data
    # ------------------------------------------
    X_train_full = X[train_indices]
    y_train_full = y[train_indices]

    X_test = X[test_indices]
    y_test = y[test_indices]

    # ------------------------------------------
    # Step 2.2: create validation set from training only
    # ------------------------------------------
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=validation_size_inside_training,
        stratify=y_train_full,
        random_state=random_seed
    )

    # ------------------------------------------
    # Step 2.3: create amplitude-scaled training data only
    # ------------------------------------------
    X_train_augmented = apply_amplitude_scaling(
        X_train,
        min_scale=min_scale,
        max_scale=max_scale
    )
    y_train_augmented = y_train.copy()

    # ------------------------------------------
    # Step 2.4: combine original and augmented training data
    # ------------------------------------------
    X_train_combined = np.concatenate([X_train, X_train_augmented], axis=0)
    y_train_combined = np.concatenate([y_train, y_train_augmented], axis=0)

    # ------------------------------------------
    # Step 2.5: normalize using combined training data only
    # ------------------------------------------
    training_mean = np.mean(X_train_combined)
    training_std = np.std(X_train_combined)

    if training_std == 0:
        training_std = 1.0

    X_train_combined = (X_train_combined - training_mean) / training_std
    X_val = (X_val - training_mean) / training_std
    X_test = (X_test - training_mean) / training_std

    # ------------------------------------------
    # Step 2.6: add final dimension for CNN input
    # ------------------------------------------
    X_train_combined = X_train_combined[..., np.newaxis]
    X_val = X_val[..., np.newaxis]
    X_test = X_test[..., np.newaxis]

    # ------------------------------------------
    # Step 2.7: convert labels to one-hot format
    # ------------------------------------------
    y_train_combined_one_hot = to_categorical(y_train_combined, num_classes=number_of_classes)
    y_val_one_hot = to_categorical(y_val, num_classes=number_of_classes)
    y_test_one_hot = to_categorical(y_test, num_classes=number_of_classes)

    # ------------------------------------------
    # Step 2.8: build a fresh baseline EEGNet model
    # ------------------------------------------
    input_shape = (number_of_channels, number_of_samples_per_trial, 1)

    model = build_baseline_eegnet(
        input_shape=input_shape,
        number_of_classes=number_of_classes
    )

    print("==============================================")
    print("Running amplitude-scaling fold:", fold_number)
    print("Original training size:", len(X_train))
    print("Augmented training size:", len(X_train_augmented))
    print("Combined training shape:", X_train_combined.shape)
    print("Validation shape:", X_val.shape)
    print("Test shape:", X_test.shape)
    print("==============================================")

    # ------------------------------------------
    # Step 2.9: train the model
    # ------------------------------------------
    history = model.fit(
        X_train_combined,
        y_train_combined_one_hot,
        epochs=number_of_epochs,
        batch_size=batch_size,
        validation_data=(X_val, y_val_one_hot),
        verbose=1
    )

    # ------------------------------------------
    # Step 2.10: evaluate on the test fold
    # ------------------------------------------
    test_loss, test_accuracy = model.evaluate(
        X_test,
        y_test_one_hot,
        verbose=0
    )

    print("Amplitude-scaling fold", fold_number, "test accuracy:", test_accuracy)
    print("==============================================")

    result_dictionary = {
        "fold_number": fold_number,
        "original_train_size": int(len(X_train)),
        "augmented_train_size": int(len(X_train_augmented)),
        "combined_train_size": int(len(X_train_combined)),
        "validation_size": int(len(X_val)),
        "test_size": int(len(X_test)),
        "test_loss": float(test_loss),
        "test_accuracy": float(test_accuracy)
    }

    return result_dictionary, history

In [ ]:

"""
This will be almost the same as the Gaussian-noise version.
The only difference is:
  -> instead of making a noisy copy of X_train
  -> we make an amplitude-scaled copy of X_train

Everything else stays the same:
  -> validation untouched
  -> test untouched
  -> baseline EEGNet unchanged
  -> same fold logic

Part 1
It creates:
  -> training data
  -> test data

Part 2
It creates:
  -> training subset
  -> validation subset

Part 3
It takes only the training subset and makes a scaled copy:
  -> X_train_augmented
That means:
  -> same trial
  -> same label
  -> slightly weaker or stronger amplitude

Part 4
It combines:
  -> original training data
  -> amplitude-scaled training data
So training size doubles.

Part 5
It normalizes using the combined training data only.

Part 6
It trains the same baseline EEGNet.

So again:
  -> model stays same
  -> only augmentation changes

That makes the experiment fair.

"""



'\nThis will be almost the same as the Gaussian-noise version.\nThe only difference is:\n  -> instead of making a noisy copy of X_train\n  -> we make an amplitude-scaled copy of X_train\n\nEverything else stays the same:\n  -> validation untouched\n  -> test untouched\n  -> baseline EEGNet unchanged\n  -> same fold logic\n\nPart 1\nIt creates:\n  -> training data\n  -> test data\n\nPart 2\nIt creates:\n  -> training subset\n  -> validation subset\n\nPart 3\nIt takes only the training subset and makes a scaled copy:\n  -> X_train_augmented\nThat means:\n  -> same trial\n  -> same label\n  -> slightly weaker or stronger amplitude\n\nPart 4\nIt combines:\n  -> original training data\n  -> amplitude-scaled training data\nSo training size doubles.\n\nPart 5\nIt normalizes using the combined training data only.\n\nPart 6\nIt trains the same baseline EEGNet.\n\nSo again:\n  -> model stays same \n  -> only augmentation changes\n\nThat makes the experiment fair.\n\n'

In [ ]:
# Run only Fold 1 first with amplitude scaling

# ==============================================
# Phase 4 - Step 3: test only Fold 1 first
# with amplitude scaling augmentation
# ==============================================

first_train_indices, first_test_indices = fold_splits[0]

amplitude_fold_1_result, amplitude_fold_1_history = run_one_fold_with_amplitude_scaling(
    X=X,
    y=y,
    train_indices=first_train_indices,
    test_indices=first_test_indices,
    fold_number=1,
    min_scale=0.9,
    max_scale=1.1
)

print("Amplitude-scaling Fold 1 finished.")
print(amplitude_fold_1_result)

Running amplitude-scaling fold: 1
Original training size: 224
Augmented training size: 224
Combined training shape: (448, 6, 4096, 1)
Validation shape: (56, 6, 4096, 1)
Test shape: (71, 6, 4096, 1)
Epoch 1/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 22s 689ms/step - accuracy: 0.2478 - loss: 2.3292 - val_accuracy: 0.0893 - val_loss: 2.3903
Epoch 2/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 13s 417ms/step - accuracy: 0.7656 - loss: 1.7827 - val_accuracy: 0.1071 - val_loss: 2.3785
Epoch 3/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 12s 423ms/step - accuracy: 0.8795 - loss: 1.5143 - val_accuracy: 0.1071 - val_loss: 2.3676
Epoch 4/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 19s 352ms/step - accuracy: 0.9085 - loss: 1.3857 - val_accuracy: 0.1786 - val_loss: 2.3516
Epoch 5/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 12s 422ms/step - accuracy: 0.9040 - loss: 1.3327 - val_accuracy: 0.1607 - val_loss: 2.3522
Epoch 6/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 12s 421ms/step - accuracy: 0.9196 - loss: 1.2750 - val_accuracy: 0.1429 - val_loss: 2.3668
Epoch 7/50
28/28 ━━━━━━━━━━━

In [ ]:
"""
We test only one fold first to make sure:

augmentation is applied correctly
training size doubles correctly
shapes are correct
model trains without errors
test evaluation works

Only after that will we run all 5 folds.


The most important thing to confirm is:

original training size ≈ 224
augmented training size ≈ 224
combined training size ≈ 448

If that happens, the augmentation is working correctly.



As per the output, we can see the Fold 1 amplitude-scaling result that shows :

1. Sizes

Everything is correct:
-> Original training size: 224
-> Augmented training size: 224
-> Combined training size: 448
-> Validation size: 56
-> Test size: 71
So the augmentation pipeline is working properly.

2. Fold 1 test accuracy
WE got: 0.22535210847854614
about 22.54%
That is a very interesting result.

3. Compare Fold 1 only
-> Baseline Fold 1: about 21.13%
-> Gaussian-noise Fold 1: about 21.13%
-> Amplitude-scaling Fold 1: about 22.54%
So for this one fold, amplitude scaling looks a little better.
But very important:
  ==> we still should not conclude from one fold alone
  ==> The real comparison must come from all 5 folds.

"""


'\nWe test only one fold first to make sure:\n\naugmentation is applied correctly\ntraining size doubles correctly\nshapes are correct\nmodel trains without errors\ntest evaluation works\n\nOnly after that will we run all 5 folds.\n\n\nThe most important thing to confirm is:\n\noriginal training size ≈ 224\naugmented training size ≈ 224\ncombined training size ≈ 448\n\nIf that happens, the augmentation is working correctly.\n\n\n\nAs per the output, we can see the Fold 1 amplitude-scaling result that shows : \n\n1. Sizes\n\nEverything is correct:\n-> Original training size: 224\n-> Augmented training size: 224\n-> Combined training size: 448\n-> Validation size: 56\n-> Test size: 71\nSo the augmentation pipeline is working properly.\n\n2. Fold 1 test accuracy\nWE got: 0.22535210847854614\nabout 22.54%\nThat is a very interesting result.\n\n3. Compare Fold 1 only\n-> Baseline Fold 1: about 21.13%\n-> Gaussian-noise Fold 1: about 21.13%\n-> Amplitude-scaling Fold 1: about 22.54%\nSo for 

In [ ]:
# ==============================================
# Phase 4 - Step 4: run all 5 folds
# with amplitude scaling augmentation
# ==============================================

all_amplitude_fold_results = []
all_amplitude_fold_histories = []

for fold_number, (train_indices, test_indices) in enumerate(fold_splits, start=1):
    fold_result, fold_history = run_one_fold_with_amplitude_scaling(
        X=X,
        y=y,
        train_indices=train_indices,
        test_indices=test_indices,
        fold_number=fold_number,
        min_scale=0.9,
        max_scale=1.1
    )

    all_amplitude_fold_results.append(fold_result)
    all_amplitude_fold_histories.append(fold_history)

print("==============================================")
print("All 5 amplitude-scaling folds finished.")
print("==============================================")

Running amplitude-scaling fold: 1
Original training size: 224
Augmented training size: 224
Combined training shape: (448, 6, 4096, 1)
Validation shape: (56, 6, 4096, 1)
Test shape: (71, 6, 4096, 1)
Epoch 1/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 14s 377ms/step - accuracy: 0.2545 - loss: 2.2912 - val_accuracy: 0.1429 - val_loss: 2.3898
Epoch 2/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 11s 391ms/step - accuracy: 0.7277 - loss: 1.6825 - val_accuracy: 0.1250 - val_loss: 2.3876
Epoch 3/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 21s 421ms/step - accuracy: 0.8326 - loss: 1.4339 - val_accuracy: 0.1250 - val_loss: 2.3923
Epoch 4/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 10s 356ms/step - accuracy: 0.8594 - loss: 1.3389 - val_accuracy: 0.0893 - val_loss: 2.3992
Epoch 5/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 11s 397ms/step - accuracy: 0.8906 - loss: 1.2589 - val_accuracy: 0.1250 - val_loss: 2.3826
Epoch 6/50
28/28 ━━━━━━━━━━━━━━━━━━━━ 12s 420ms/step - accuracy: 0.9018 - loss: 1.2219 - val_accuracy: 0.1250 - val_loss: 2.3925
Epoch 7/50
28/28 ━━━━━━━━━━━

In [ ]:
# # ==============================================
# Phase 4 - Step 5: convert amplitude-scaling results into table
# ==============================================

amplitude_results_dataframe = pd.DataFrame(all_amplitude_fold_results)

print("==============================================")
print("Amplitude-scaling model fold-wise results:")
print("==============================================")
display(amplitude_results_dataframe)

Amplitude-scaling model fold-wise results:


,fold_number,original_train_size,augmented_train_size,combined_train_size,validation_size,test_size,test_loss,test_accuracy
0,1,224,224,448,56,71,2.590365,0.197183
1,2,224,224,448,57,70,2.589665,0.142857
2,3,224,224,448,57,70,2.801848,0.085714
3,4,224,224,448,57,70,2.627474,0.157143
4,5,224,224,448,57,70,2.702040,0.171429


In [ ]:
# Compute final mean and standard deviation


# ==============================================
# Phase 4 - Step 6: calculate final mean and standard deviation
# ==============================================

amplitude_mean_accuracy = amplitude_results_dataframe["test_accuracy"].mean()
amplitude_std_accuracy = amplitude_results_dataframe["test_accuracy"].std()

print("==============================================")
print("Final BASELINE EEGNet + Amplitude Scaling result")
print("==============================================")
print("Fold accuracies:", amplitude_results_dataframe["test_accuracy"].tolist())
print("Mean accuracy:", amplitude_mean_accuracy)
print("Standard deviation:", amplitude_std_accuracy)
print("Mean ± Std:", str(round(amplitude_mean_accuracy, 4)) + " ± " + str(round(amplitude_std_accuracy, 4)))
print("==============================================")

Final BASELINE EEGNet + Amplitude Scaling result
Fold accuracies: [0.19718310236930847, 0.1428571492433548, 0.08571428805589676, 0.15714286267757416, 0.17142857611179352]
Mean accuracy: 0.15086519569158555
Standard deviation: 0.041582387406720905
Mean ± Std: 0.1509 ± 0.0416


In [ ]:
"""
Final comparison so far
1) Baseline EEGNet -->> 12.82% ± 2.67%
2) EEGNet + Attention -->> 12.53% ± 3.68%
3) EEGNet + Gaussian Noise -->> 14.80% ± 4.06%
4) EEGNet + Amplitude Scaling -->> 15.09% ± 4.16%


current best result is:
1. Baseline EEGNet + Amplitude Scaling -->> 15.09% ± 4.16%
That means amplitude scaling is slightly better than Gaussian noise.
Improvement over baseline
  -->> Baseline: 12.82%
  -->> Amplitude scaling: 15.09%

Improvement: about 2.27 percentage points

That is small, but in this difficult low-data imagined-speech EEG setting, it is still a real gain.

Ranking from best to worst
    ==> EEGNet + Amplitude Scaling → 15.09% ± 4.16%
    ==> EEGNet + Gaussian Noise → 14.80% ± 4.06%
    ==> Baseline EEGNet → 12.82% ± 2.67%
    ==> EEGNet + Attention → 12.53% ± 3.68%

Important research insights are :

Architectural change alone did not help
    -> Attention did not improve the result.

Training-data expansion helped
  -> Both augmentation methods improved performance.

And among them:

Amplitude scaling helped the most so far
  `-> So results are clearly pointing in one direction:

for this problem, improving the training data is more useful than adding attention alone
That is a strong research finding.
But also be honest

Even the best result is still only: 15.09%

So the problem is not solved.
That means:
-> overfitting is still present
-> augmentation helps, but only modestly
-> stronger strategies are still needed



solid result statement is :


*** Baseline EEGNet achieved 12.82% ± 2.67% under subject-specific 5-fold cross-validation.
Adding a lightweight attention mechanism did not improve performance,
giving 12.53% ± 3.68%.
In contrast, data augmentation improved generalization:
Gaussian noise reached 14.80% ± 4.06%, and amplitude scaling achieved the best
performance at 15.09% ± 4.16%.
This suggests that, in the current low-data setting, augmentation is more
beneficial than attention-based architectural modification. ***



Problem : Small imagined-speech EEG data causes severe overfitting.
Baseline : Weak performance.
Attention : No improvement.
Augmentation : Helps.

Best current augmentation : Amplitude scaling.

So now, my research paper is naturally moving towards :
  -> robust augmentation + transfer learning


NEXT :

The best next step is: testing one more augmentation before transfer learning

recommendation : Time shifting / jittering

because:
-> it is still simpler than FTSurrogate
-> it gives you a third augmentation comparison
-> it strengthens your experimental section

So augmentation comparison table would become:
-> baseline
-> Gaussian noise
-> amplitude scaling
-> time shifting

That is much better for publication.




"""